# Forest Inspection - Evaluation Notebook

Danh gia model: metrics, per-sequence analysis, deforestation analysis.

# === ENVIRONMENT DETECTION & DATA CONFIG ===
import os, requests, zipfile
from pathlib import Path
from tqdm import tqdm

IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    DATA_ROOTS = [
        Path('/kaggle/input/uav-forest-sunny-part1'),
        Path('/kaggle/input/uav-forest-sunny-next'),
        Path('/kaggle/working/forest_sunny'),
    ]
else:
    DATA_ROOTS = [Path('data/forest_sunny')]

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZENODO_BASE = 'https://zenodo.org/records/15511426/files'

def download_seq(seq_num, target_dir):
    seq_dir = target_dir / f'seq{seq_num}'
    if seq_dir.exists() and len(list(seq_dir.rglob('*.png'))) > 0:
        print(f'[OK] seq{seq_num} da co san')
        return
    url = f'{ZENODO_BASE}/seq{seq_num}.zip?download=1'
    print(f'[DOWNLOAD] seq{seq_num}...')
    resp = requests.get(url, stream=True, timeout=60)
    total = int(resp.headers.get('content-length', 0))
    zip_path = target_dir / f'seq{seq_num}.zip'
    target_dir.mkdir(parents=True, exist_ok=True)
    with open(zip_path, 'wb') as f:
        with tqdm(total=total, unit='B', unit_scale=True, desc=f'seq{seq_num}') as pbar:
            for chunk in resp.iter_content(8192):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    print(f'[EXTRACT] seq{seq_num}...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(target_dir)
    zip_path.unlink()
    print(f'[OK] seq{seq_num} done')

# Download seq9 for test (only ~4GB)
if IS_KAGGLE:
    download_seq(9, Path('/kaggle/working/forest_sunny'))
else:
    for seq in [1, 2, 3, 4]:
        download_seq(seq, DATA_ROOTS[0])

print(f'\n[DONE] DATA_ROOTS = {DATA_ROOTS}')

In [ ]:
# === INLINE SOURCE CODE ===
import numpy as np
import time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from PIL import Image
from typing import Optional, Dict, List, Tuple, Union
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# --- Constants ---
LABEL_COLORS = [
    (0, 255, 255), (0, 127, 0), (19, 132, 69), (0, 53, 65),
    (130, 76, 0), (152, 251, 152), (151, 126, 171), (250, 150, 0),
    (115, 176, 195), (123, 123, 123), (0, 0, 0),
]
CLASS_NAMES = [
    'Sky', 'Deciduous_trees', 'Coniferous_trees', 'Fallen_trees',
    'Dirt_ground', 'Ground_vegetation', 'Rocks', 'Building',
    'Fence', 'Car', 'Empty',
]
NUM_CLASSES = len(CLASS_NAMES)
SEQUENCE_INFO = {
    'seq1': {'pitch': 0, 'altitude': 30},
    'seq2': {'pitch': -60, 'altitude': 30},
    'seq3': {'pitch': -90, 'altitude': 30},
    'seq4': {'pitch': 0, 'altitude': 50},
    'seq5': {'pitch': -60, 'altitude': 50},
    'seq6': {'pitch': -90, 'altitude': 50},
    'seq7': {'pitch': 0, 'altitude': 80},
    'seq8': {'pitch': -60, 'altitude': 80},
    'seq9': {'pitch': -90, 'altitude': 80},
}

# --- Dataset (multi-root + double-nesting) ---
def rgb_to_class_id(label_rgb):
    h, w, _ = label_rgb.shape
    class_map = np.full((h, w), NUM_CLASSES - 1, dtype=np.int64)
    for cid, color in enumerate(LABEL_COLORS):
        mask = np.all(label_rgb == np.array(color, dtype=np.uint8), axis=-1)
        class_map[mask] = cid
    return class_map

def class_id_to_rgb(class_map):
    h, w = class_map.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cid, color in enumerate(LABEL_COLORS):
        rgb[class_map == cid] = color
    return rgb

def _find_seq_dir(roots, seq):
    for root in roots:
        flat = root / seq
        if (flat / 'color').exists(): return flat
        nested = root / seq / seq
        if (nested / 'color').exists(): return nested
    return None

class ForestDataset(Dataset):
    def __init__(self, root, sequences, transform=None):
        if isinstance(root, (str, Path)): self.roots = [Path(root)]
        else: self.roots = [Path(r) for r in root]
        self.transform = transform
        self.samples = []
        for seq in sequences:
            seq_dir = _find_seq_dir(self.roots, seq)
            if seq_dir is None:
                print(f'Warning: {seq} not found, skipping.')
                continue
            for cf in sorted((seq_dir / 'color').glob('*.png')):
                lf = seq_dir / 'labels' / cf.name
                if lf.exists():
                    self.samples.append((cf, lf))
        print(f'ForestDataset: {len(self.samples)} samples from {sequences}')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        cp, lp = self.samples[idx]
        image = np.array(Image.open(cp).convert('RGB'))
        label_rgb = np.array(Image.open(lp).convert('RGB'))
        mask = rgb_to_class_id(label_rgb)
        if self.transform:
            t = self.transform(image=image, mask=mask)
            image, mask = t['image'], t['mask']
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask).long()
        return image, mask.long()

def get_val_transforms(img_size=(512, 512)):
    return A.Compose([
        A.Resize(height=img_size[0], width=img_size[1]),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_available_sequences(data_roots):
    if isinstance(data_roots, (str, Path)): data_roots = [data_roots]
    found = set()
    for root in data_roots:
        root = Path(root)
        if not root.exists(): continue
        for d in root.iterdir():
            if not d.is_dir() or not d.name.startswith('seq'): continue
            if (d / 'color').exists() or (d / d.name / 'color').exists():
                found.add(d.name)
    return sorted(found)

# --- Model ---
class UNet(nn.Module):
    def __init__(self, encoder='resnet34', encoder_weights='imagenet', num_classes=11, activation=None):
        super().__init__()
        self.model = smp.Unet(encoder_name=encoder, encoder_weights=encoder_weights,
                              in_channels=3, classes=num_classes, activation=activation)
        self.encoder_name = encoder
        self.num_classes = num_classes
    def forward(self, x): return self.model(x)

MODEL_REGISTRY = {
    'unet': lambda **kw: UNet(**kw),
    'unetpp': lambda encoder='resnet34', encoder_weights='imagenet', num_classes=11, **kw:
        smp.UnetPlusPlus(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=num_classes),
    'deeplabv3plus': lambda encoder='resnet50', encoder_weights='imagenet', num_classes=11, **kw:
        smp.DeepLabV3Plus(encoder_name=encoder, encoder_weights=encoder_weights, in_channels=3, classes=num_classes),
}
def build_model(name, **kwargs):
    if name not in MODEL_REGISTRY:
        raise ValueError(f'Unknown model: {name}. Available: {list(MODEL_REGISTRY.keys())}')
    return MODEL_REGISTRY[name](**kwargs)

# --- Metrics ---
class SegmentationMetrics:
    def __init__(self, num_classes, class_names=None):
        self.num_classes = num_classes
        self.class_names = class_names or [str(i) for i in range(num_classes)]
        self.confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    def reset(self):
        self.confusion_matrix = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)
    @torch.no_grad()
    def update(self, preds, targets):
        p, t = preds.cpu().numpy().flatten(), targets.cpu().numpy().flatten()
        valid = (t >= 0) & (t < self.num_classes)
        p, t = p[valid], t[valid]
        cm = np.bincount(t * self.num_classes + p, minlength=self.num_classes**2)
        self.confusion_matrix += cm.reshape(self.num_classes, self.num_classes)
    def compute(self):
        cm = self.confusion_matrix
        inter = np.diag(cm)
        union = cm.sum(1) + cm.sum(0) - inter
        iou = np.where(union > 0, inter / union, 0.0)
        class_total = cm.sum(axis=1)
        class_acc = np.where(class_total > 0, inter / class_total, 0.0)
        pixel_acc = inter.sum() / cm.sum() if cm.sum() > 0 else 0.0
        valid = union > 0
        miou = iou[valid].mean() if valid.any() else 0.0
        result = {'miou': float(miou), 'pixel_accuracy': float(pixel_acc),
                  'mean_class_accuracy': float(class_acc[valid].mean()) if valid.any() else 0.0}
        for i, name in enumerate(self.class_names):
            result[f'iou_{name}'] = float(iou[i])
        return result
    def get_confusion_matrix(self):
        return self.confusion_matrix.copy()

# --- Evaluator ---
class Evaluator:
    def __init__(self, model, test_loader, device='cuda', num_classes=NUM_CLASSES):
        self.model = model.to(device)
        self.test_loader = test_loader
        self.device = device
        self.num_classes = num_classes
        self.metrics = SegmentationMetrics(num_classes, CLASS_NAMES)
        self.results = {}

    @torch.no_grad()
    def evaluate(self, use_amp=True):
        self.model.eval()
        self.metrics.reset()
        inference_times = []
        for images, masks in tqdm(self.test_loader, desc='Evaluating'):
            images, masks = images.to(self.device), masks.to(self.device)
            start_t = time.time()
            with autocast(device_type='cuda', enabled=use_amp):
                outputs = self.model(images)
            if self.device == 'cuda': torch.cuda.synchronize()
            inference_times.append(time.time() - start_t)
            self.metrics.update(outputs.argmax(dim=1), masks)
        self.results = self.metrics.compute()
        self.results['avg_inference_time_ms'] = np.mean(inference_times) * 1000
        self.results['total_samples'] = len(self.test_loader.dataset)
        self.results['fps'] = len(self.test_loader.dataset) / sum(inference_times)
        return self.results

    def evaluate_per_sequence(self, dataset_roots, sequences, transform, batch_size=8):
        per_seq = {}
        for seq in sequences:
            ds = ForestDataset(root=dataset_roots, sequences=[seq], transform=transform)
            loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
            self.metrics.reset()
            self.model.eval()
            with torch.no_grad():
                for images, masks in loader:
                    images, masks = images.to(self.device), masks.to(self.device)
                    self.metrics.update(self.model(images).argmax(dim=1), masks)
            per_seq[seq] = self.metrics.compute()
            print(f'  {seq}: mIoU = {per_seq[seq]["miou"]:.4f}')
        return per_seq

    def save_predictions(self, output_dir='outputs/predictions', num_samples=20):
        self.model.eval()
        out_path = Path(output_dir)
        out_path.mkdir(parents=True, exist_ok=True)
        count = 0
        mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
        with torch.no_grad():
            for images, masks in self.test_loader:
                preds = self.model(images.to(self.device)).argmax(dim=1).cpu().numpy()
                for i in range(images.shape[0]):
                    if count >= num_samples: return
                    img = images[i].cpu().numpy().transpose(1, 2, 0)
                    img = ((img * std + mean) * 255).clip(0, 255).astype(np.uint8)
                    visualize_prediction(img, masks[i].numpy(), preds[i],
                                       save_path=str(out_path / f'pred_{count:04d}.png'),
                                       title=f'Sample {count}')
                    count += 1
        print(f'Saved {count} predictions to {output_dir}')

    def print_results(self):
        if not self.results:
            print('No results. Run evaluate() first.')
            return
        print('\n' + '='*55)
        print('  EVALUATION RESULTS')
        print('='*55)
        print(f'  mIoU:             {self.results.get("miou", 0):.4f}')
        print(f'  Pixel Accuracy:   {self.results.get("pixel_accuracy", 0):.4f}')
        print(f'  Mean Class Acc:   {self.results.get("mean_class_accuracy", 0):.4f}')
        print(f'  Avg Inference:    {self.results.get("avg_inference_time_ms", 0):.1f} ms')
        print(f'  FPS:              {self.results.get("fps", 0):.1f}')
        print('-'*55)
        print('  Per-Class IoU:')
        for name in CLASS_NAMES:
            iou = self.results.get(f'iou_{name}', 0)
            bar = '#' * int(iou * 30)
            print(f'    {name:20s} {iou:.4f} {bar}')
        print('='*55)

# --- Visualization ---
def visualize_prediction(image, gt_mask, pred_mask, save_path=None, title=''):
    gt_rgb = class_id_to_rgb(gt_mask)
    pred_rgb = class_id_to_rgb(pred_mask)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image if image.max() > 1 else (image * 255).astype(np.uint8))
    axes[0].set_title('Input Image'); axes[0].axis('off')
    axes[1].imshow(gt_rgb); axes[1].set_title('Ground Truth'); axes[1].axis('off')
    axes[2].imshow(pred_rgb); axes[2].set_title('Prediction'); axes[2].axis('off')
    patches = [mpatches.Patch(color=np.array(c)/255, label=n) for n, c in zip(CLASS_NAMES, LABEL_COLORS)]
    fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=8)
    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

# --- Deforestation Analysis ---
TREE_CLASSES = {1: 'Deciduous_trees', 2: 'Coniferous_trees'}
FALLEN_TREE_CLASSES = {3: 'Fallen_trees'}
VEGETATION_CLASSES = {5: 'Ground_vegetation'}
GROUND_CLASSES = {4: 'Dirt_ground', 6: 'Rocks'}
STRUCTURE_CLASSES = {7: 'Building', 8: 'Fence', 9: 'Car'}
IGNORE_CLASSES = {0: 'Sky', 10: 'Empty'}

def compute_deforestation_index(pred_mask):
    total = pred_mask.size
    tree_px = sum((pred_mask == c).sum() for c in TREE_CLASSES)
    fallen_px = sum((pred_mask == c).sum() for c in FALLEN_TREE_CLASSES)
    veg_px = sum((pred_mask == c).sum() for c in VEGETATION_CLASSES)
    ground_px = sum((pred_mask == c).sum() for c in GROUND_CLASSES)
    struct_px = sum((pred_mask == c).sum() for c in STRUCTURE_CLASSES)
    ignore_px = sum((pred_mask == c).sum() for c in IGNORE_CLASSES)
    eff = max(total - ignore_px, 1)
    all_trees = tree_px + fallen_px
    return {
        'canopy_cover': float(tree_px / eff),
        'fallen_tree_ratio': float(fallen_px / all_trees) if all_trees > 0 else 0.0,
        'vegetation_index': float((tree_px + veg_px) / eff),
        'deforestation_degree': 1.0 - float(tree_px / eff),
        'disturbance_index': float((fallen_px + ground_px + struct_px) / eff),
    }

@torch.no_grad()
def analyze_sequence_deforestation(model, dataloader, device='cuda', sequence_name=''):
    model.eval()
    all_idx = []
    for images, _ in tqdm(dataloader, desc=f'Analyzing {sequence_name}'):
        preds = model(images.to(device)).argmax(dim=1).cpu().numpy()
        for i in range(preds.shape[0]):
            all_idx.append(compute_deforestation_index(preds[i]))
    agg = {}
    for key in all_idx[0]:
        vals = [d[key] for d in all_idx]
        agg[f'mean_{key}'] = float(np.mean(vals))
        agg[f'std_{key}'] = float(np.std(vals))
    return agg

def plot_deforestation_comparison(per_seq, save_path=None):
    seqs = list(per_seq.keys())
    metrics = ['mean_canopy_cover', 'mean_deforestation_degree', 'mean_disturbance_index']
    labels = ['Canopy Cover', 'Deforestation Degree', 'Disturbance Index']
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, metric, label in zip(axes, metrics, labels):
        vals = [per_seq[s].get(metric, 0) for s in seqs]
        colors = plt.cm.RdYlGn_r(np.array(vals))
        bars = ax.bar(seqs, vals, color=colors)
        ax.set_title(label, fontsize=12, fontweight='bold')
        ax.set_ylabel('Value'); ax.set_ylim(0, 1)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{val:.2f}', ha='center', fontsize=9)
    plt.suptitle('Deforestation Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[OK] All modules loaded. Device: {device}')

In [ ]:
# === DATA SETUP ===
# Kaggle: add 2 datasets vao notebook input:
#   1. leehien12/uav-forest-sunny-part1 (seq1-4)
#   2. khesng/uav-forest-sunny-next (seq5-8)

import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle')

def find_seq_dirs(root):
    """Tim tat ca thu muc seq co chua color/ ben trong, bat ke do sau."""
    found = {}
    for p in sorted(root.rglob('color')):
        seq_dir = p.parent  # thu muc cha cua color/
        if seq_dir.name.startswith('seq'):
            found[seq_dir.name] = seq_dir
    return found

if IS_KAGGLE:
    DATA_ROOT = Path('/kaggle/working/data')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    
    for inp in [Path('/kaggle/input/datasets/leehien12/uav-forest-sunny-part1'),
                Path('/kaggle/input/datasets/khesng/uav-forest-sunny-next')]:
        if not inp.exists():
            print(f'[WARNING] Not found: {inp.name}')
            print(f'  -> Add Data > Search: "{inp.name}"')
            continue
        seq_dirs = find_seq_dirs(inp)
        for name, real_path in seq_dirs.items():
            link = DATA_ROOT / name
            if not link.exists():
                os.symlink(real_path, link)
                print(f'[OK] {name} -> {real_path}')
else:
    DATA_ROOT = Path('data/forest_sunny')

# Verify
seqs = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir() and d.name.startswith('seq')])
print(f'\nDATA_ROOT: {DATA_ROOT}')
print(f'Sequences: {seqs} ({len(seqs)} total)')
for s in seqs:
    c = len(list((DATA_ROOT / s / 'color').glob('*.png'))) if (DATA_ROOT / s / 'color').exists() else 0
    print(f'  {s}: {c} images')
OUTPUT_DIR = Path('/kaggle/working/outputs') if IS_KAGGLE else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

In [ ]:
split = {
    'train': ['seq1', 'seq2', 'seq3', 'seq4', 'seq5', 'seq7', 'seq8'],
    'val': ['seq6'],
    'test': ['seq9'],
}
val_tf = get_val_transforms(img_size=(512, 512))

test_ds = ForestDataset(DATA_ROOTS, split['test'], transform=val_tf)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

print(f'Test set: {len(test_ds)} samples from {split["test"]}')

evaluator = Evaluator(model, test_loader, device=device)
results = evaluator.evaluate()
evaluator.print_results()

avail_seqs = get_available_sequences(DATA_ROOTS)
per_seq = evaluator.evaluate_per_sequence(DATA_ROOTS, avail_seqs, val_tf, batch_size=8)

fig, ax = plt.subplots(figsize=(10, 5))
seq_names = list(per_seq.keys())
miou_vals = [per_seq[s]['miou'] for s in seq_names]
colors = plt.cm.RdYlGn(np.array(miou_vals))
bars = ax.bar(seq_names, miou_vals, color=colors)
ax.set_ylabel('mIoU')
ax.set_title('mIoU per Sequence', fontsize=14)
ax.set_ylim(0, 1)
for bar, val in zip(bars, miou_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}',
            ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
MODEL_NAME = 'unet'
ENCODER = 'resnet34'

# Tim checkpoint moi nhat
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_PATH = None

if CHECKPOINT_DIR.exists():
    ckpt_files = sorted(CHECKPOINT_DIR.glob('*.pth'), key=lambda f: f.stat().st_mtime, reverse=True)
    if ckpt_files:
        CHECKPOINT_PATH = str(ckpt_files[0])
        print(f'Found {len(ckpt_files)} checkpoints. Using: {ckpt_files[0].name}')

if not CHECKPOINT_PATH:
    CHECKPOINT_PATH = str(OUTPUT_DIR / 'checkpoints' / 'best_model.pth')
    print(f'No checkpoints found. Expected at: {CHECKPOINT_PATH}')

model = build_model(MODEL_NAME, encoder=ENCODER, num_classes=NUM_CLASSES)

if Path(CHECKPOINT_PATH).exists():
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded checkpoint: {CHECKPOINT_PATH}')
    print(f'  Epoch: {ckpt.get("epoch", "?")}, mIoU: {ckpt.get("metric", 0):.4f}')
else:
    print('[WARNING] Checkpoint not found. Using random weights (pipeline test only).')
    print('  -> Hay chay 02_train.ipynb truoc de co model weights.')

# Uu tien top-down views (pitch -90 deg)
deforest_seqs = [s for s in avail_seqs if SEQUENCE_INFO.get(s, {}).get('pitch') == -90]
if not deforest_seqs:
    deforest_seqs = avail_seqs[:2]

print(f'Analyzing deforestation for: {deforest_seqs}')
deforest_results = {}
for seq in deforest_seqs:
    ds = ForestDataset(DATA_ROOTS, [seq], transform=val_tf)
    loader = DataLoader(ds, batch_size=8, shuffle=False)
    deforest_results[seq] = analyze_sequence_deforestation(model, loader, device, seq)

plot_deforestation_comparison(deforest_results, save_path=str(OUTPUT_DIR / 'deforestation_analysis.png'))

In [ ]:
split = get_split_for_available(str(DATA_ROOT))
val_tf = get_val_transforms(img_size=(512, 512))

test_ds = ForestDataset(str(DATA_ROOT), split['test'], transform=val_tf)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

print(f'Test set: {len(test_ds)} samples from {split["test"]}')

evaluator = Evaluator(model, test_loader, device=device)
results = evaluator.evaluate()
evaluator.print_results()

## 3. Per-Sequence Analysis

In [ ]:
avail_seqs = get_available_sequences(str(DATA_ROOT))
per_seq = evaluator.evaluate_per_sequence(str(DATA_ROOT), avail_seqs, val_tf, batch_size=8)

fig, ax = plt.subplots(figsize=(10, 5))
seq_names = list(per_seq.keys())
miou_vals = [per_seq[s]['miou'] for s in seq_names]
colors = plt.cm.RdYlGn(np.array(miou_vals))
bars = ax.bar(seq_names, miou_vals, color=colors)
ax.set_ylabel('mIoU')
ax.set_title('mIoU per Sequence', fontsize=14)
ax.set_ylim(0, 1)
for bar, val in zip(bars, miou_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}',
            ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Prediction Visualization

In [ ]:
evaluator.save_predictions(output_dir=str(OUTPUT_DIR / 'predictions'), num_samples=10)

## 5. Deforestation Analysis

In [ ]:
# Uu tien top-down views (pitch -90 deg)
deforest_seqs = [s for s in avail_seqs if SEQUENCE_INFO.get(s, {}).get('pitch') == -90]
if not deforest_seqs:
    deforest_seqs = avail_seqs[:2]

print(f'Analyzing deforestation for: {deforest_seqs}')
deforest_results = {}
for seq in deforest_seqs:
    ds = ForestDataset(str(DATA_ROOT), [seq], transform=val_tf)
    loader = DataLoader(ds, batch_size=8, shuffle=False)
    deforest_results[seq] = analyze_sequence_deforestation(model, loader, device, seq)

plot_deforestation_comparison(deforest_results, save_path=str(OUTPUT_DIR / 'deforestation_analysis.png'))

## 6. Confusion Matrix

In [ ]:
import seaborn as sns

cm = evaluator.metrics.get_confusion_matrix()
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Normalized Confusion Matrix', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()